In [0]:
df = spark.table("predictstockprices.plstocks.silver_financial_ratios")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load the tables
financial_ratios = spark.table("predictstockprices.plstocks.silver_financial_ratios")
sector_lookup = spark.table("predictstockprices.plstocks.silver_sector_lookup")

# Join the tables on the 'ticker' column
joined_df = financial_ratios.join(sector_lookup, on="ticker", how="left").drop("insert_timestamp")

# Reorder columns to have 'sector' as the second column
cols = joined_df.columns
cols.remove('sector')
new_col_order = [cols[0], 'sector'] + cols[1:]
joined_df = joined_df.select(new_col_order)

# Define the financial metric columns to calculate sector averages for
value_columns = [
    "price_book_value", "price_graham_book_value", "price_sales", "price_earnings",
    "price_operating_profit", "ev_sales", "ev_ebit", "ev_ebitda"
]

# Calculate sector averages (rounded to 2 decimal places), only for non-null sectors
for col in value_columns:
    avg_col_name = f"{col}_sector_avg"
    avg_expr = F.round(F.avg(col).over(Window.partitionBy("sector")), 2)
    joined_df = joined_df.withColumn(
        avg_col_name,
        F.when(F.col("sector").isNotNull(), avg_expr).otherwise(None)
    )

# Build column order: ticker | sector | metric | sector_avg | ...
base_cols = ["ticker", "sector"]
interleaved_cols = []
for col in value_columns:
    interleaved_cols.extend([col, f"{col}_sector_avg"])

# Apply the new column order to the DataFrame
final_col_order = base_cols + interleaved_cols
joined_df = joined_df.select(final_col_order)


In [0]:
joined_df.write.format("delta").mode("overwrite").saveAsTable("plstocks.gold_financial_health_score")